# 🎙️ Chạy Text-To-Speech & Voice Cloning trên Google Colab

> **Lưu ý quan trọng:** Hãy đảm bảo bạn đã bật **T4 GPU** để mô hình sinh giọng nói nhanh nhất:
> - Vào menu: **Runtime** -> **Change runtime type**
> - Tại ô **Hardware accelerator**, chọn **T4 GPU**
> - Bấm **Save**


In [ ]:
#@title 1. Kiểm tra GPU và Tải mã nguồn
!nvidia-smi

import os
if not os.path.exists('/content/text_to_speech'):
    !git clone https://github.com/doptcennos/text_to_speech.git /content/text_to_speech
else:
    %cd /content/text_to_speech
    !git pull

%cd /content/text_to_speech


In [ ]:
#@title 2. Cài đặt thư viện hệ thống và Python (mất khoảng 1 - 2 phút)
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip install -q -r backend/requirements.txt
print('✅ Đã cài đặt xong tất cả thư viện cần thiết!')


In [ ]:
#@title 3. Cài đặt Cloudflare Tunnel (Miễn phí, không cần tài khoản)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print('✅ Cloudflared Tunnel đã sẵn sàng!')


In [ ]:
#@title 4. Khởi động Web & Backend Text-To-Speech (Bấm chạy và click vào link)
import subprocess
import time
import re

# Dừng tiến trình cũ nếu có
!pkill -f uvicorn > /dev/null 2>&1
!pkill -f cloudflared > /dev/null 2>&1

print('🚀 Đang khởi động Backend FastAPI...')
backend_proc = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/text_to_speech/backend'
)

time.sleep(6)

print('🌐 Đang tạo đường link truy cập công khai qua Cloudflare Tunnel...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

print('\n' + '='*65)
print('🔍 ĐANG TẠO LINK TRUY CẬP...')
print('='*65)

for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print(f'\n🎉 ĐÃ KHỞI CHẠY THÀNH CÔNG! BẤM VÀO LINK DƯỚI ĐỂ DÙNG:\n')
        print(f'👉 {public_url} 👈\n')
        print('='*65)
        break

try:
    backend_proc.wait()
except KeyboardInterrupt:
    print('\n🛑 Đang dừng server...')
    backend_proc.terminate()
    tunnel_proc.terminate()
